In [17]:
import os
import glob
import gc
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, IterableDataset
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# Check for CUDA
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [18]:
class ResidualBlock(nn.Module):
    def __init__(self,inplanes ,planes,shortcut = None,downsample = False,*args, **kwargs):
        super().__init__()
        if downsample == True:
            self.stride = 2
        else:
            self.stride = 1 
        self.layer1 = nn.Conv2d(inplanes,planes,3,self.stride,padding=1)
        self.activation = nn.ReLU()
        self.layer2 = nn.Conv2d(in_channels = planes,out_channels= planes,kernel_size=3,stride =1,padding=1)
        self.shortcut = shortcut
        self.downsample = downsample
        self.norm1 = nn.BatchNorm2d(planes)
        self.norm2 = nn.BatchNorm2d(planes)

    def forward(self,x) :
        shortcut = 0
        if self.shortcut:
            shortcut = self.shortcut(x)
        x = self.layer1(x)
        x = self.norm1(x)
        x = self.activation(x)
        x = self.layer2(x)
        x = self.norm2(x)
        return self.activation(x+shortcut)

class Bottleneck(nn.Module):
    def __init__(self,inplanes,planes,downsample = False,shortcut = None):
        super().__init__()
        self.stride = None
        if downsample == True:
            self.stride = 2
        else :
            self.stride = 1
        self.layer1 = nn.Conv2d(inplanes,planes,1,self.stride,padding=0)
        self.layer2 = nn.Conv2d(planes,planes,3,1,padding=1)
        self.layer3 = nn.Conv2d(planes,planes,1,padding=0)
        self.shortcut  = shortcut
        self.activation  = nn.ReLU()
        self.norm1 = nn.BatchNorm2d(planes)
        self.norm2 = nn.BatchNorm2d(planes)
    def forward(self,x):
        shortcut = 0 
        if self.shortcut:
            shortcut = self.shortcut(x)
        x = self.layer1(x)
        x = self.norm1(x)
        x = self.activation(x)
        x = self.layer2(x)
        x = self.norm2(x)
        x = self.activation(x)
        x = self.layer3(x)
        x = x+shortcut
        return self.activation(x)

class Upscale(nn.Module):
    def __init__(self,inplanes,planes = None,expansion =2, *args, **kwargs):
        super().__init__()
        if planes:
            self.upscale = nn.ConvTranspose2d(inplanes,planes,kernel_size=2,stride=2)
        else:
            self.upscale = nn.ConvTranspose2d(inplanes,inplanes*expansion,kernel_size=2,stride=2)

        self.norm = nn.BatchNorm2d(inplanes*2)
        self.activation = nn.ReLU()
        self.expansion = expansion
    def forward(self,x):
        x = self.upscale(x)
        x = self.norm(x)
        return self.activation(x)

class convnet(nn.Module):
    def __init__(self, layers: list, inplanes, expansion=2):
        super().__init__()
        self.no_blocks_in_layers = layers
        self.expansion = expansion
        
        self.inplanes = 64 #used for book keeping such that it keep track of the planes of the prev layer
        # Output calculation: Floor((125 + 2*2 - 3)/2 + 1) = Floor(126/2 + 1) = 64
        self.layer1 = nn.Sequential(
                nn.Conv2d(in_channels=inplanes, out_channels=64, kernel_size=3, stride=2, padding=2),
                nn.BatchNorm2d(64),
            ) # the first ever layer
        self.layers = nn.ModuleList()
        self.globalavgpool = nn.AdaptiveAvgPool2d((1,1))
        self.build_layers()
        
    @property
    def no_of_layers(self):
        return len(self.no_blocks_in_layers)
    
    def make_layer_Res(self,no_of_blocks,planes,downsample):
        blocks = []
        if downsample == True:
            blocks.append(ResidualBlock(self.inplanes,planes,downsample=True,shortcut=nn.Conv2d(self.inplanes,planes,1,2)))
        for i in range(no_of_blocks-1):
            blocks.append(ResidualBlock(planes,planes,shortcut=nn.Identity()))
        self.inplanes = planes
        return nn.Sequential(*blocks)
    
    def make_layer_Botl(self,no_of_blocks,planes,downsample):
        blocks = []
        if downsample == True:
            blocks.append(Bottleneck(self.inplanes,planes,downsample=True,shortcut=nn.Conv2d(self.inplanes,planes,1,2)))
        for i in range(no_of_blocks-1):
            blocks.append(Bottleneck(planes,planes,shortcut=nn.Identity()))
        self.inplanes = planes
        return nn.Sequential(*blocks)
    
    def build_layers(self):
        blocks_count = 0
        total_blocks = sum(self.no_blocks_in_layers)
        
        # We apply bottlenecks when layers > 50 (approx 24 blocks), roughly after halfway
        # Alternatively, we just use a hardcoded threshold
        bottleneck_treshold = total_blocks // 2 if total_blocks >= 24 else float('inf')
        
        for i in range(self.no_of_layers):
            if i == 0:
                self.layers.append(self.make_layer_Res(self.no_blocks_in_layers[i],self.inplanes,False))
                blocks_count += self.no_blocks_in_layers[i]
                continue 
            if blocks_count < bottleneck_treshold :
                self.layers.append(self.make_layer_Res(self.no_blocks_in_layers[i],self.inplanes*self.expansion,True))
            else:
                self.layers.append(self.make_layer_Botl(self.no_blocks_in_layers[i],self.inplanes*self.expansion,True))
            blocks_count += self.no_blocks_in_layers[i]

    def forward(self,x) :
        x = self.layer1(x)
        for i in self.layers:
            x = i(x)
        x = self.globalavgpool(x)
        return x


In [19]:
# 2. Initialize Model, Optimizer, and Scheduler
# Changed inplanes to 4 based on requirement "use 4 channels at most".
# Our Dataset class now ensures all inputs are exactly 4 channels.


class JetMassModel(nn.Module):

    def __init__(self, layers, in_channels, expansion=1):
        super().__init__()

        # backbone
        self.backbone = convnet(
            layers=layers,
            inplanes=in_channels,
            expansion=expansion
        )

        # determine feature size produced by backbone
        self.feature_dim = self.backbone.inplanes

        # regression head
        self.head = nn.Linear(self.feature_dim, 1)


    def forward(self, x):

        x = self.backbone(x)     # (B,C,1,1)

        x = torch.flatten(x, 1)  # (B,C)

        x = self.head(x)         # (B,1)
        return x*128+290
model = JetMassModel(
    layers=[4,6,7,6],
    in_channels=4,
    expansion=1
)
criterion = nn.MSELoss()
initial_lr = 3e-4
# model = model.to(device)
# if torch.cuda.device_count()>1:
#     model = nn.DataParallel(model)
optimizer = optim.Adam(model.parameters(), lr=initial_lr,weight_decay = 1e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=100,
    eta_min=1e-5
)

print("Model Initialized with 4 input channels.")
print(f"Initial LR: {optimizer.param_groups[0]['lr']}")

Model Initialized with 4 input channels.
Initial LR: 0.0003


In [20]:
!pip install onnx

In [21]:
checkpoint_path = "latest_checkpoint_file_201.pth"

state = torch.load(checkpoint_path, map_location=device)

# remove DataParallel prefix
clean_state = {k.replace("module.", ""): v for k, v in state.items()}

# load into underlying model
if isinstance(model, torch.nn.DataParallel):
    model.module.load_state_dict(clean_state)
else:
    model.load_state_dict(clean_state)

print("Checkpoint loaded correctly")
model.eval()

# dummy input with correct shape
dummy_input = torch.randn(1, 4, 32, 32)

torch.onnx.export(
    model,
    dummy_input,
    "sample.onnx",
    input_names=["input"],
    output_names=["output"],
    opset_version=11
)

print("Model exported to sample.onnx")

Checkpoint loaded correctly
Model exported to sample.onnx
